# MLP (Multi-Layer Perceptron) - Neural Network Dasar
**Repositori**: Machine Learning
**Topik**: Implementasi MLPClassifier untuk klasifikasi kondisi medis
**Dataset**: medical_conditions_dataset.csv
---
**Pendahuluan**: MLP adalah arsitektur neural network dasar yang terdiri dari input layer, hidden layer(s), dan output layer. Backpropagation digunakan untuk mengupdate bobot.


## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


## 2. Load & EDA


In [ ]:
df = pd.read_csv('../../data/medical_conditions_dataset.csv')
print('Shape:', df.shape)
print(df.head())
print(df['condition'].value_counts())
plt.figure(figsize=(10, 5))
sns.countplot(y='condition', data=df, order=df['condition'].value_counts().index)
plt.title('Distribusi Kondisi Medis')
plt.tight_layout()
plt.show()


## 3. Data Preparation


In [ ]:
df_clean = df.drop(['id', 'full_name'], axis=1)
for col in ['gender', 'smoking_status']:
    df_clean[col] = LabelEncoder().fit_transform(df_clean[col])
X = df_clean.drop('condition', axis=1)
y = df_clean['condition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')


## 4. MLPClassifier - Default


In [ ]:
mlp_default = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
mlp_default.fit(X_train_scaled, y_train)
y_pred = mlp_default.predict(X_test_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))


## 5. MLP dengan Berbagai Arsitektur


In [ ]:
architectures = [('(50,)', 50), ('(100,)', 100), ('(50, 25)', (50, 25)), ('(100, 50)', (100, 50))]
results = []
for name, hls in architectures:
    mlp = MLPClassifier(hidden_layer_sizes=hls, max_iter=500, random_state=42)
    mlp.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, mlp.predict(X_test_scaled))
    results.append({'Arsitektur': name, 'Accuracy': acc})
    print(f'{name}: {acc:.4f}')
pd.DataFrame(results).set_index('Arsitektur').plot(kind='bar', legend=False)
plt.title('Perbandingan Arsitektur MLP')
plt.ylabel('Accuracy')
plt.tight_layout()
plt.show()


## 6. Hyperparameter Tuning


In [ ]:
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (50, 25)],
    'activation': ['relu', 'tanh'],
    'learning_rate_init': [0.001, 0.01]
}
grid = GridSearchCV(MLPClassifier(max_iter=500, random_state=42), param_grid, cv=3, scoring='accuracy', verbose=1)
grid.fit(X_train_scaled, y_train)
print('Best params:', grid.best_params_)
print('Best CV score:', grid.best_score_)
y_pred_best = grid.predict(X_test_scaled)
print('Test accuracy:', accuracy_score(y_test, y_pred_best))


## 7. Loss Curve


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(grid.best_estimator_.loss_curve_, label='Training Loss', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Loss Curve - MLP Terbaik')
plt.legend()
plt.grid(True)
plt.show()


## 8. Confusion Matrix


In [ ]:
cm = confusion_matrix(y_test, grid.predict(X_test_scaled))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples')
plt.title('Confusion Matrix - MLP Terbaik')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


## 9. Kesimpulan
MLP mampu menangkap pola non-linear kompleks pada data medis. Arsitektur dengan 1-2 hidden layer sudah cukup untuk dataset berukuran sedang. Tuning hyperparameter seperti learning rate dan aktivasi sangat mempengaruhi performa.
